# Meta-RL LSTM Değerlendirme ve Düzeltme

Bu notebook, **RecurrentPPO (SB3)** ile **Meta-RL (RL²)** tarzı değerlendirmede, epizotlar arası **LSTM hafızasını taşımayı** (carry-over) doğru şekilde yapan bir çerçeve sunar.

**Öne çıkanlar**
- `eval_rl2_on_task(...)` ile *carry_memory=True* olduğunda yeni epizotta `episode_start=False` kullanılır ve **LSTM resetlenmez**.
- *carry_memory=False* iken SB3'nin varsayılanı gibi `episode_start=True` ile **reset** edilir.
- Boyut ve tutarlılık kontrolleri, env vektörlemesi ve SB3 RNN arayüzü için güvenlik kontrolleri içerir.


In [ ]:
# === 0) Kurulum ve importlar ===
import numpy as np
from typing import Optional, Tuple, List

try:
    import gymnasium as gym
except Exception:
    import gym

from stable_baselines3.common.vec_env import DummyVecEnv, VecEnv
from stable_baselines3.common.utils import obs_as_tensor
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3 import PPO
from stable_baselines3.common.type_aliases import GymObs

try:
    # Recurrent policy (SB3 contrib)
    from sb3_contrib import RecurrentPPO
except Exception as e:
    RecurrentPPO = None
    print("Uyarı: sb3_contrib bulunamadı. RecurrentPPO kullanmak için 'pip install sb3-contrib' gereklidir.")


In [ ]:
# === 1) Yardımcı: VecEnv inşa (örnek/şablon) ===
def make_env(env_id: str, seed: int = 0):
    def _thunk():
        env = gym.make(env_id)
        env.reset(seed=seed)
        return env
    return _thunk

def build_vec_env(env_id: str, n_envs: int = 1) -> VecEnv:
    return DummyVecEnv([make_env(env_id, seed=i) for i in range(n_envs)])


In [ ]:
# === 2) Güvenlik ve tutarlılık kontrolleri ===
def check_model_is_recurrent(model) -> None:
    """Recurrent policy mi? predict arayüzü LSTM state alıyor mu?"""
    if not hasattr(model, 'predict'):
        raise TypeError("Model SB3 arayüzünü uygulamıyor (predict yok).")
    if RecurrentPPO is not None and isinstance(model, RecurrentPPO):
        return
    # Bazı custom sınıflar RecurrentPPO dışı olabilir, predict(state=...) desteği aranır
    try:
        # Dry-run arg isimlerini kontrol edelim
        model.predict(None, state=None, episode_start=None, deterministic=True)
    except TypeError as e:
        msg = (
            "Model görünüşe göre recurrent değil ya da (state, episode_start) argümanlarını desteklemiyor.\n"
            "RecurrentPPO veya predict(state=..., episode_start=...) desteği olan bir model kullanın.\n"
            f"Detay: {e}"
        )
        raise TypeError(msg)
    except Exception:
        # None obs ile predict çağırmak hata verebilir; sadece imza testi yaptık.
        pass

def check_vecenv_shapes(env: VecEnv):
    assert isinstance(env, VecEnv), "Env bir VecEnv instance olmalı (DummyVecEnv gibi)."
    assert env.num_envs >= 1, "VecEnv içinde en az bir alt-çevre olmalı."


In [ ]:
# === 3) Meta-RL Değerlendirme: LSTM hafızasını epizotlar arasında taşıma kontrolü ===
def eval_rl2_on_task(
    model,
    env: VecEnv,
    n_episodes: int = 5,
    deterministic: bool = True,
    carry_memory: bool = True,
    max_steps_per_episode: Optional[int] = None,
) -> List[float]:
    """
    Meta-RL (RL²) değerlendirme:
    - carry_memory=True: Yeni epizotta LSTM resetlenmez (episode_start=False), state taşınır.
    - carry_memory=False: SB3 klasik; yeni epizotta LSTM resetlenir (episode_start=True, state=None).

    Dönüş: epizot toplam getirileri listesi (len = n_episodes)
    """
    check_model_is_recurrent(model)
    check_vecenv_shapes(env)
    assert env.num_envs == 1, "Bu değerlendirme tek-env içindir (n_envs=1)."  # RL² analizi için basit tutuyoruz

    # SB3 RNN arayüzü: lstm_states = (h, c) tuple'ları içeren np.array'ler olabilir.
    lstm_states = None
    episode_starts = np.array([True], dtype=bool)  # İlk adımda True fark etmez; reset ile eşleşir.

    ep_returns: List[float] = []
    obs = env.reset()
    # SB3: değerlendirmede train modunu kapatmak iyi bir pratiktir.
    if hasattr(model, 'policy') and hasattr(model.policy, 'set_training_mode'):
        model.policy.set_training_mode(False)

    for ep in range(n_episodes):
        ep_ret = 0.0
        steps = 0

        # === Yeni epizot başı: hafıza stratejisi ===
        if carry_memory:
            # LSTM state'i *koruyoruz* ve SB3'e reset işareti vermiyoruz
            episode_starts[:] = False
        else:
            # LSTM state'i sıfırla (klasik)
            lstm_states = None
            episode_starts[:] = True

        # SB3 RNN predict döngüsü
        while True:
            action, lstm_states = model.predict(
                obs, state=lstm_states, episode_start=episode_starts, deterministic=deterministic
            )
            obs, reward, done, infos = env.step(action)
            ep_ret += float(reward[0])
            steps += 1

            # Yeni adımlar artık epizot içindedir
            episode_starts[:] = False

            if max_steps_per_episode is not None and steps >= max_steps_per_episode:
                # Zorlama bitiş: env.reset çağır ve döngüyü kır
                obs = env.reset()
                break

            if done[0]:
                # Env kapandı (time-limit veya başarısızlık), *sonraki epizoda* hazırlan
                obs = env.reset()
                break

        ep_returns.append(ep_ret)

    return ep_returns


In [ ]:
# === 4) Hızlı Sağlamlık Testleri (isteğe bağlı) ===
def _dry_signature_test(model, env):
    # Basit bir adım çalışsın
    obs = env.reset()
    episode_starts = np.array([True], dtype=bool)
    lstm_states = None
    try:
        action, lstm_states = model.predict(obs, state=lstm_states, episode_start=episode_starts, deterministic=True)
        assert action is not None
        return True
    except Exception as e:
        print("Predict dry-run hatası:", e)
        return False


In [ ]:
# === 5) Örnek kullanım (model yükleme size bağlı) ===
EXAMPLE = False  # Bir checkpoint yolunuz varsa True yapıp deneyebilirsiniz.
MODEL_PATH = "path/to/your/recurrentppo.zip"  # örnek
ENV_ID = "CartPole-v1"  # kendi IsaacLab veya custom env'inizi buraya koyabilirsiniz

if EXAMPLE and RecurrentPPO is not None:
    env = build_vec_env(ENV_ID, n_envs=1)
    model = RecurrentPPO.load(MODEL_PATH, env=env, device="auto")
    ok = _dry_signature_test(model, env)
    if ok:
        print("Dry-run OK. RL² değerlendirme başlıyor...")
        print("carry_memory=True:")
        print(eval_rl2_on_task(model, env, n_episodes=5, deterministic=True, carry_memory=True))
        print("carry_memory=False:")
        print(eval_rl2_on_task(model, env, n_episodes=5, deterministic=True, carry_memory=False))
    else:
        print("Model/policy arayüzünde sorun var.")


## Notlar ve Öneriler
- SB3 RNN arayüzünde **`episode_start=True`** verildiğinde LSTM **resetlenir**. Meta-RL'de epizotlar arası adaptasyonu ölçmek istiyorsanız yeni epizoda geçerken **`episode_start=False`** kullanın ve **`lstm_states`**'i **taşıyın**.
- Eğer farklı **görev/task**'lara geçiyorsanız, *carry_memory=True* ile politika aynı iç durumu taşıyacağından **hızlı adaptasyon** etkisini gözleyebilirsiniz.
- Değerlendirme boyunca `model.policy.set_training_mode(False)` çağrısı ile dropout/normalization gibi eğitim davranışlarını kapattık.
- `env.num_envs == 1` kısıtı basitlik içindir. İsterseniz vektörlü değerlendirme için `done`/`episode_starts` maskelemesini genişletebilirsiniz.
